# Проверка bbox-разметки

Создадим проект, обучение, экзамен и основной пул для проверки рамок.
Каждый полный запуск создаёт новый проект.

Рядом с тетрадкой должна лежать папка `review_task_files`.


In [1]:
%pip install -q "requests>=2.32,<3"



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## API

Токен берём из `YSDA_BUTTERFLIES_TOKEN` или вводим вручную.


In [2]:
import json
import os
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone
from getpass import getpass
from pathlib import Path

import requests

TOKEN = os.environ.get("YSDA_BUTTERFLIES_TOKEN") or getpass("Токен Яндекс Заданий: ")
BASE_URL = "https://tasks.yandex.ru/api/v1"
headers = {"Authorization": f"OAuth {TOKEN}", "Content-Type": "application/json"}


def request(method, url, **kwargs):
    response = requests.request(method, url, headers=headers, timeout=60, **kwargs)
    if not response.ok:
        message = response.text.replace(TOKEN, "[скрыто]")[:1500]
        raise RuntimeError(f"{method} {url}: HTTP {response.status_code}: {message}")
    return response.json()


def get_items(path, **params):
    rows = []
    while True:
        page = request(
            "GET", BASE_URL + path, params={**params, "limit": 100, "sort": "id"}
        )
        rows.extend(page["items"])
        if not page["has_more"]:
            return rows
        params["id_gt"] = page["items"][-1]["id"]


## Настройки

Каждый ответ проверяют три человека. На странице два обычных фото и один ханипот, оплата — 0,5 ₽.
Для 400 ответов: 400 / 2 × 3 × 0,5 = 300 ₽ без комиссии и повторных проверок.


In [ ]:
DATA_DIR = Path("review_task_files")
RESULTS = Path("results/review")
PROJECT_NAME = "Проверка разметки бабочек"
BBOX_PROJECT_ID = "11760"
BBOX_POOL_ID = "6866092"
BBOX_PARTICIPATION_SKILL_ID = "21403"
TRAINING_PER_PAGE = 5
EXAM_PER_PAGE = 5
EXAM_PASS_PERCENT = 80
REAL_PER_PAGE = 2
GOLDEN_PER_PAGE = 1
REWARD_PER_PAGE = 0.5
OVERLAP = 3
TIME_LIMIT_SECONDS = 1800
QUALITY_WINDOW = 5
QUALITY_PASS_PERCENT = 80
QUALITY_BAN_DAYS = 7
FAST_PAGE_SECONDS = 4
FAST_PAGES_LIMIT = 2
will_expire = (datetime.now(timezone.utc) + timedelta(days=30)).strftime(
    "%Y-%m-%dT%H:%M:%S"
)


## Участники bbox-проекта

После первой принятой страницы выдаём навык участия со значением 1.
Он нужен, чтобы участники разметки не получали задания на проверку.


In [4]:
participation_rule = {
    "collector_config": {"type": "ANSWER_COUNT"},
    "rules": [
        {
            "conditions": [
                {"key": "assignments_accepted_count", "operator": "GTE", "value": 1}
            ],
            "action": {
                "type": "SET_SKILL",
                "parameters": {
                    "skill_id": BBOX_PARTICIPATION_SKILL_ID,
                    "skill_value": 1,
                },
            },
        }
    ],
}
bbox_project = request("GET", f"{BASE_URL}/projects/{BBOX_PROJECT_ID}")
bbox_payload = {
    key: value
    for key, value in bbox_project.items()
    if key not in ["id", "status", "owner", "created"]
}
configs = bbox_payload.setdefault("quality_control", {}).setdefault("configs", [])
configs[:] = [
    config
    for config in configs
    if not any(
        rule["action"].get("parameters", {}).get("skill_id")
        == BBOX_PARTICIPATION_SKILL_ID
        for rule in config["rules"]
    )
]
configs.append(participation_rule)
request("PUT", f"{BASE_URL}/projects/{BBOX_PROJECT_ID}", json=bbox_payload)


{'id': '11760',
 'public_name': 'Разметка бабочек — проверка IoU',
 'public_description': 'Выделите всех взрослых бабочек, включая мотыльков и моль, прямоугольными рамками.',
 'public_instructions': '<h2>Выделите всех взрослых бабочек на фотографии</h2>\n<p>Обведите каждую взрослую бабочку отдельной прямоугольной рамкой.\nРазмечайте дневных и ночных бабочек, включая мотыльков и моль.</p>\n<h3>Правила разметки</h3>\n<ul><li>Найдите всех бабочек, включая маленьких, частично закрытых и находящихся у края фотографии.</li><li>Плотно охватите все видимые части: крылья, тело, голову, усики и лапки.\nНе обрезайте их и не добавляйте большой запас фона.</li><li>Для частично закрытой бабочки выделяйте видимые части одной рамкой.\nНе достраивайте скрытые части; фон между видимыми частями допустим.</li><li>Если бабочка выходит за кадр, рамка заканчивается на границе фотографии.</li><li>Одна бабочка — одна рамка. Рамки разных бабочек могут пересекаться.\nНе объединяйте несколько бабочек одной рамкой

### Кто уже участвовал

Выдадим тот же навык участникам, которые уже проходили обучение, экзамен или разметку.


In [5]:
bbox_pools = get_items("/pools", project_id=BBOX_PROJECT_ID)
bbox_trainings = get_items("/trainings", project_id=BBOX_PROJECT_ID)
participant_ids = set()
for pool in bbox_pools + bbox_trainings:
    assignments = get_items("/assignments", pool_id=pool["id"])
    participant_ids.update(
        a["user_id"]
        for a in assignments
        if a["status"] in ["ACCEPTED", "SUBMITTED", "REJECTED"]
    )
for training in bbox_trainings:
    participant_ids.update(
        row["user_id"]
        for row in get_items("/user-skills", skill_id=training["skill_id"])
    )
for user_id in sorted(participant_ids):
    request(
        "PUT",
        BASE_URL + "/user-skills",
        json={
            "skill_id": BBOX_PARTICIPATION_SKILL_ID,
            "user_id": user_id,
            "value": 1,
        },
    )
print("Отмечено участников bbox:", len(participant_ids))


Отмечено участников bbox: 0


## Данные

Из 40 эталонных фото: 5 в обучении, 5 в экзамене, 30 в ханипотах.
Разбиение такое же, как в bbox-проекте. Фото с тремя бабочками — в обучении.

В части примеров специально удалены, добавлены или обрезаны рамки.
Эталон содержит все ошибки, которые нужно выбрать.


In [6]:
def read_json(name):
    return json.loads((DATA_DIR / name).read_text(encoding="utf-8"))


training_tasks = read_json("training_5.json")
exam_tasks = read_json("exam_5.json")
honeypots = read_json("honeypots_30.json")
source_photos = {row["photo_id"]: row for row in read_json("source_photos_400.json")}
print(
    {
        "Обучение": len(training_tasks),
        "Экзамен": len(exam_tasks),
        "Ханипоты": len(honeypots),
    }
)


{'Обучение': 5, 'Экзамен': 5, 'Ханипоты': 30}


### Загрузка заданий


In [7]:
def upload_tasks(pool_id, tasks):
    for start in range(0, len(tasks), 50):
        batch = [dict(task, pool_id=pool_id) for task in tasks[start : start + 50]]
        response = request(
            "POST",
            BASE_URL + "/tasks",
            json=batch,
            params={
                "async_mode": "false",
                "skip_invalid_items": "false",
                "allow_defaults": "true",
            },
        )
        if response.get("validation_errors"):
            raise RuntimeError(
                json.dumps(response["validation_errors"], ensure_ascii=False)
            )
    print("Пул", pool_id, "— загружено:", len(tasks))


## Интерфейс

Рамки можно посмотреть, но нельзя изменить. Ошибок можно выбрать несколько.
«Всё верно» выбирается отдельно. Клавиши 1–4 переключают варианты ответа.


### Варианты ответа


In [8]:
answer_options = [
    {"label": "Всё верно", "value": "correct"},
    {"label": "Пропущена бабочка", "value": "missing"},
    {"label": "Лишняя рамка", "value": "extra"},
    {"label": "Неточные границы", "value": "bounds"},
]


### Проверка выбора

Нужен хотя бы один ответ. «Всё верно» нельзя выбрать вместе с ошибками.


In [9]:
answer_schema = {
    "type": "object",
    "properties": {
        "correct": {"type": "boolean"},
        "missing": {"type": "boolean"},
        "extra": {"type": "boolean"},
        "bounds": {"type": "boolean"},
    },
    "required": ["correct", "missing", "extra", "bounds"],
    "additionalProperties": False,
    "oneOf": [
        {
            "properties": {
                "correct": {"enum": [True]},
                "missing": {"enum": [False]},
                "extra": {"enum": [False]},
                "bounds": {"enum": [False]},
            }
        },
        {
            "properties": {"correct": {"enum": [False]}},
            "anyOf": [
                {"properties": {"missing": {"enum": [True]}}},
                {"properties": {"extra": {"enum": [True]}}},
                {"properties": {"bounds": {"enum": [True]}}},
            ],
        },
    ],
}


### Просмотр рамок и хоткеи


In [10]:
REVIEW_CONFIG = {
    "view": {
        "type": "view.list",
        "items": [
            {
                "type": "view.markdown",
                "content": "**Проверьте всю фотографию и предложенные рамки.**",
            },
            {
                "type": "field.image-annotation",
                "image": {"type": "data.input", "path": "image"},
                "data": {
                    "type": "data.internal",
                    "path": "display_boxes",
                    "default": {"type": "data.input", "path": "proposed_boxes"},
                },
                "disabled": True,
                "shapes": {"rectangle": True},
                "fullHeight": False,
                "ratio": [16, 9],
                "minWidth": 150,
            },
            {
                "type": "field.checkbox-group",
                "label": "Отметьте все найденные ошибки. «Всё верно» выбирайте отдельно.",
                "data": {
                    "type": "data.output",
                    "path": "category",
                    "default": {
                        "correct": False,
                        "missing": False,
                        "extra": False,
                        "bounds": False,
                    },
                },
                "preserveFalse": True,
                "options": answer_options,
                "validation": {
                    "type": "condition.schema",
                    "schema": answer_schema,
                    "hint": "Выберите «Всё верно» или одну/несколько ошибок.",
                },
            },
        ],
    },
    "plugins": [
        {"type": "plugin.toloka", "layout": {"kind": "scroll", "taskWidth": 800}},
        {
            "type": "plugin.hotkeys",
            "1": {
                "type": "action.toggle",
                "data": {"type": "data.output", "path": "category.correct"},
            },
            "2": {
                "type": "action.toggle",
                "data": {"type": "data.output", "path": "category.missing"},
            },
            "3": {
                "type": "action.toggle",
                "data": {"type": "data.output", "path": "category.extra"},
            },
            "4": {
                "type": "action.toggle",
                "data": {"type": "data.output", "path": "category.bounds"},
            },
        },
    ],
}


### Версии компонентов


In [11]:
REVIEW_LOCK = {
    "core": "1.16.51",
    "plugin.hotkeys": "1.3.52",
    "condition.schema": "1.1.58",
    "field.image-annotation": "1.5.56",
    "plugin.toloka": "1.1.61",
    "action.toggle": "1.0.57",
    "view.markdown": "1.0.60",
    "view.list": "1.2.52",
    "field.checkbox-group": "1.2.64",
}


### Инструкция исполнителю


In [12]:
INSTRUCTIONS = r"""
<h2>Проверьте разметку бабочек</h2>
<p>На фотографии уже предложены прямоугольные рамки. Проверьте весь кадр и каждую рамку. Менять рамки здесь нельзя.</p>
<p>Правила те же, что в bbox-проекте: размечаем всех взрослых дневных и ночных бабочек, включая мотыльков и моль. Каждая бабочка выделяется отдельно. Рамка плотно охватывает все видимые крылья, тело, усики и лапки. Скрытые части не достраиваются. Других насекомых, гусениц, куколок, тени, рисунки и отражения не выделяем.</p>
<h3>Варианты ответа</h3>
<ul><li><b>1 — Всё верно.</b> Все бабочки выделены, лишних рамок нет, границы плотные. Если бабочек нет и рамок тоже нет — это верная разметка.</li>
<li><b>2 — Пропущена бабочка.</b> Есть взрослая бабочка, которой не досталось отдельной рамки. Одна общая рамка на нескольких бабочках тоже считается пропуском отдельных объектов.</li>
<li><b>3 — Лишняя рамка.</b> Рамка выделяет не бабочку, фон или повторно выделяет уже размеченную бабочку.</li>
<li><b>4 — Неточные границы.</b> Рамка обрезает видимые части бабочки или захватывает большой лишний запас фона.</li></ul>
<p>Отметьте все найденные виды ошибок. «Всё верно» выбирайте только отдельно. Каждое нажатие клавиши 1–4 включает или выключает вариант. Удерживать клавишу не нужно.</p>
<p>Используйте масштабирование и перемещение изображения. Просмотрите весь кадр: бабочки могут быть маленькими, перекрывать друг друга и находиться у края.</p>
<p>Если фотография не загрузилась или вы не можете уверенно оценить ответ, пропустите задание средствами платформы. Не отмечайте «Всё верно» вместо проблемы загрузки.</p>
"""


## Проект review


In [13]:
input_spec = {
    "photo_id": {"type": "string", "required": True, "hidden": True},
    "image": {"type": "url", "required": True, "hidden": False},
    "proposed_boxes": {"type": "array_json", "required": True, "hidden": False},
    "annotation_id": {"type": "string", "required": True, "hidden": True},
    "source_assignment_id": {"type": "string", "required": False, "hidden": True},
    "source_task_id": {"type": "string", "required": False, "hidden": True},
    "source_worker_id": {"type": "string", "required": False, "hidden": True},
}
project_payload = {
    "public_name": PROJECT_NAME,
    "public_description": "Проверьте, что все бабочки выделены и рамки поставлены правильно.",
    "public_instructions": INSTRUCTIONS,
    "assignments_issuing_type": "AUTOMATED",
    "assignments_automerge_enabled": False,
    "task_spec": {
        "input_spec": input_spec,
        "output_spec": {
            "category": {"type": "json", "required": True, "hidden": False}
        },
        "view_spec": {
            "type": "tb",
            "config": json.dumps(REVIEW_CONFIG, ensure_ascii=False),
            "lock": REVIEW_LOCK,
            "inferDataSpec": False,
            "settings": {
                "showSkip": True,
                "showTimer": True,
                "showSubmit": True,
                "showInstructions": True,
                "showFullscreen": True,
            },
        },
    },
}
project = request("POST", BASE_URL + "/projects", json=project_payload)
project_id = str(project["id"])
print(f"https://tasks.yandex.ru/requester/project/{project_id}")


https://tasks.yandex.ru/requester/project/11776


## Навык экзамена

В нём хранится процент правильных ответов. Для допуска нужно не меньше 80%.


In [14]:
review_skill = request(
    "POST",
    BASE_URL + "/skills",
    json={
        "name": f"Бабочки: экзамен review {project_id}",
        "hidden": True,
    },
)
review_skill_id = str(review_skill["id"])


## Обучение

Пять фото на одной странице, с подсказками при ошибке.


In [15]:
training = request(
    "POST",
    BASE_URL + "/trainings",
    json={
        "project_id": project_id,
        "private_name": "Review: обучение, 5 фото",
        "inherited_instructions": True,
        "may_contain_adult_content": False,
        "assignment_max_duration_seconds": TIME_LIMIT_SECONDS,
        "mix_tasks_in_creation_order": True,
        "shuffle_tasks_in_task_suite": False,
        "training_tasks_in_task_suite_count": TRAINING_PER_PAGE,
        "task_suites_required_to_pass": 1,
    },
)
training_pool_id = str(training["id"])
upload_tasks(training_pool_id, training_tasks)


Пул 6866271 — загружено: 5


## Экзамен

Пять фото на одной странице. После неё записываем результат в навык экзамена.


In [16]:
review_filters = [
    {"category": "profile", "key": "languages", "operator": "IN", "value": "RU"},
    {
        "category": "computed",
        "key": "client_type",
        "operator": "EQ",
        "value": "BROWSER",
    },
    {
        "category": "skill",
        "key": BBOX_PARTICIPATION_SKILL_ID,
        "operator": "EQ",
        "value": None,
    },
]
exam_payload = {
    "project_id": project_id,
    "private_name": "Review: экзамен, 5 фото",
    "type": "REGULAR",
    "may_contain_adult_content": False,
    "reward_per_assignment": 0.0,
    "assignment_max_duration_seconds": TIME_LIMIT_SECONDS,
    "auto_accept_solutions": True,
    "will_expire": will_expire,
    "defaults": {
        "default_overlap_for_new_tasks": 10,
        "default_overlap_for_new_task_suites": 10,
    },
    "mixer_config": {
        "real_tasks_count": 0,
        "golden_tasks_count": EXAM_PER_PAGE,
        "training_tasks_count": 0,
        "min_real_tasks_count": 0,
        "min_golden_tasks_count": EXAM_PER_PAGE,
        "force_last_assignment": False,
    },
    "filter": {"and": review_filters},
    "quality_control": {
        "training_requirement": {
            "training_pool_id": training_pool_id,
            "training_passing_skill_value": 0,
        },
        "configs": [
            {
                "collector_config": {"type": "GOLDEN_SET", "parameters": {}},
                "rules": [
                    {
                        "conditions": [
                            {
                                "key": "golden_set_answers_count",
                                "operator": "GTE",
                                "value": len(exam_tasks),
                            }
                        ],
                        "action": {
                            "type": "SET_SKILL_FROM_OUTPUT_FIELD",
                            "parameters": {
                                "skill_id": review_skill_id,
                                "from_field": "golden_set_correct_answers_rate",
                            },
                        },
                    }
                ],
            }
        ],
    },
}
exam_pool = request("POST", BASE_URL + "/pools", json=exam_payload)
exam_pool_id = str(exam_pool["id"])
upload_tasks(exam_pool_id, exam_tasks)


Пул 6866272 — загружено: 5


## Основной пул

Два обычных фото и один ханипот на странице, перекрытие — 3.
Точность ниже 80% на последних пяти ханипотах закрывает доступ на 7 дней.
Страницы принимаются автоматически. Решение по рамкам соберём отдельно из ответов ревьюеров.


### Слишком быстрые ответы

Две страницы быстрее 4 секунд закрывают доступ к проекту постоянно. Порог относится ко всей странице из трёх фото.


In [ ]:
speed_rule = {
    "collector_config": {
        "type": "ASSIGNMENT_SUBMIT_TIME",
        "parameters": {"fast_submit_threshold_seconds": FAST_PAGE_SECONDS},
    },
    "rules": [
        {
            "conditions": [
                {
                    "key": "fast_submitted_count",
                    "operator": "GTE",
                    "value": FAST_PAGES_LIMIT,
                }
            ],
            "action": {
                "type": "RESTRICTION_V2",
                "parameters": {
                    "scope": "PROJECT",
                    "duration_unit": "PERMANENT",
                },
            },
        }
    ],
}


### Создание пула


In [ ]:
main_payload = {
    "project_id": project_id,
    "private_name": "Review: проверка bbox",
    "type": "REGULAR",
    "may_contain_adult_content": False,
    "reward_per_assignment": REWARD_PER_PAGE,
    "assignment_max_duration_seconds": TIME_LIMIT_SECONDS,
    "auto_accept_solutions": True,
    "will_expire": will_expire,
    "defaults": {
        "default_overlap_for_new_tasks": OVERLAP,
        "default_overlap_for_new_task_suites": OVERLAP,
    },
    "mixer_config": {
        "real_tasks_count": REAL_PER_PAGE,
        "golden_tasks_count": GOLDEN_PER_PAGE,
        "training_tasks_count": 0,
        "min_real_tasks_count": REAL_PER_PAGE,
        "min_golden_tasks_count": GOLDEN_PER_PAGE,
        "force_last_assignment": False,
    },
    "filter": {
        "and": review_filters
        + [
            {
                "category": "skill",
                "key": review_skill_id,
                "operator": "GTE",
                "value": EXAM_PASS_PERCENT,
            },
        ]
    },
    "quality_control": {
        "configs": [
            speed_rule,
            {
                "collector_config": {
                    "type": "GOLDEN_SET",
                    "parameters": {"history_size": QUALITY_WINDOW},
                },
                "rules": [
                    {
                        "conditions": [
                            {
                                "key": "golden_set_answers_count",
                                "operator": "GTE",
                                "value": QUALITY_WINDOW,
                            },
                            {
                                "key": "golden_set_correct_answers_rate",
                                "operator": "LT",
                                "value": QUALITY_PASS_PERCENT,
                            },
                        ],
                        "action": {
                            "type": "RESTRICTION_V2",
                            "parameters": {
                                "scope": "PROJECT",
                                "duration": QUALITY_BAN_DAYS,
                                "duration_unit": "DAYS",
                            },
                        },
                    }
                ],
            },
        ]
    },
}
main_pool = request("POST", BASE_URL + "/pools", json=main_payload)
main_pool_id = str(main_pool["id"])
upload_tasks(main_pool_id, honeypots)


## Задания из bbox-разметки

Берём принятые ответы на 400 обычных фото. Ханипоты пропускаем.
Один ответ исполнителя на фото — одно задание со всеми его рамками.
Разные ответы на одно фото проверяем отдельно.

Когда появятся новые ответы, повторяем подготовку и загрузку ниже.


In [18]:
def full_assignments(pool_id):
    rows = get_items("/assignments", pool_id=pool_id, status="ACCEPTED")
    assignments = []
    for row in rows:
        if "solutions" not in row:
            row = request("GET", f"{BASE_URL}/assignments/{row['id']}")
        assignments.append(row)
    return assignments


def prepare_review_tasks(assignments):
    tasks = []
    for assignment in assignments:
        for task, solution in zip(
            assignment["tasks"], assignment["solutions"], strict=True
        ):
            photo_id = task["input_values"]["photo_id"]
            if photo_id not in source_photos:
                continue
            tasks.append(
                {
                    "input_values": {
                        "photo_id": photo_id,
                        "image": task["input_values"]["image"],
                        "proposed_boxes": solution["output_values"]["result"],
                        "annotation_id": assignment["id"] + ":" + task["id"],
                        "source_assignment_id": assignment["id"],
                        "source_task_id": task["id"],
                        "source_worker_id": assignment["user_id"],
                    }
                }
            )
    return tasks


### Получение ответов


In [19]:
bbox_assignments = full_assignments(BBOX_POOL_ID)
main_tasks = prepare_review_tasks(bbox_assignments)
for user_id in sorted(
    {task["input_values"]["source_worker_id"] for task in main_tasks}
):
    request(
        "PUT",
        BASE_URL + "/user-skills",
        json={
            "skill_id": BBOX_PARTICIPATION_SKILL_ID,
            "user_id": user_id,
            "value": 1,
        },
    )
print("Обычных bbox-ответов для проверки:", len(main_tasks))


Обычных bbox-ответов для проверки: 0


### Загрузка новых ответов

Уже загруженные ответы пропускаем по `annotation_id`.


In [20]:
uploaded_ids = {
    task["input_values"]["annotation_id"]
    for task in get_items("/tasks", pool_id=main_pool_id)
}
new_tasks = [
    task
    for task in main_tasks
    if task["input_values"]["annotation_id"] not in uploaded_ids
]
upload_tasks(main_pool_id, new_tasks)


Пул 6866273 — загружено: 0


## Результаты проверки

Ждём три ответа от разных ревьюеров. Большинство «Всё верно» — принимаем разметку.
Большинство по виду ошибки — отправляем на исправление. Без большинства — разбираем вручную.


In [21]:
def decide_review(votes):
    if len(votes) < OVERLAP:
        return "pending", []
    majority = len(votes) // 2 + 1
    if sum(vote["correct"] for vote in votes) >= majority:
        return "accept", []
    errors = [
        key
        for key in ["missing", "extra", "bounds"]
        if sum(vote[key] for vote in votes) >= majority
    ]
    return ("fix", errors) if errors else ("manual", [])


### Выгрузка


In [22]:
def export_review_results():
    RESULTS.mkdir(parents=True, exist_ok=True)
    reviews = full_assignments(main_pool_id)
    sources = {
        task["input_values"]["annotation_id"]: task["input_values"]
        for task in main_tasks
    }
    by_annotation = defaultdict(dict)
    for assignment in reviews:
        for task, solution in zip(
            assignment["tasks"], assignment["solutions"], strict=True
        ):
            annotation_id = task["input_values"]["annotation_id"]
            if annotation_id in sources:
                by_annotation[annotation_id][assignment["user_id"]] = solution[
                    "output_values"
                ]["category"]
    decisions = []
    for annotation_id, source in sources.items():
        votes = by_annotation[annotation_id]
        decision, errors = decide_review(list(votes.values()))
        decisions.append(
            {**source, "decision": decision, "errors": errors, "reviews": votes}
        )
    for name, rows in [
        ("bbox_answers", bbox_assignments),
        ("review_answers", reviews),
        ("decisions", decisions),
    ]:
        with (RESULTS / (name + ".jsonl")).open("w", encoding="utf-8") as file:
            for row in rows:
                file.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(Counter(row["decision"] for row in decisions))
    return decisions


После завершения проверки: `decisions = export_review_results()`.

В `results/review` сохраняются исходные ответы, проверки и решения.
Разметку с ошибками исправляем вручную и сохраняем причину исправления.
Если на одно фото приняты разные наборы рамок, выбираем итоговый вручную.
После исправлений собираем golden set из 400 размеченных фото.
